In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

C:\Users\maxwe\AppData\Local\Temp\ipykernel_18472\1992594709.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [17]:
df = pd.read_csv('data2.csv')

In [18]:
df

,Array Size,Top Down Time (us),Bottom Up Time (us)
0,10,6,4
1,20,13,10
2,30,26,20
3,40,26,25
4,50,28,22
...,...,...,...
995,9960,6216,5580
996,9970,12570,9085
997,9980,10370,8107
998,9990,2376,1198


In [49]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=df["Array Size"], y=df["Top Down Time (us)"], mode="markers", name="Top Down"))
fig.add_trace(go.Scatter(x=df["Array Size"], y=df["Bottom Up Time (us)"], mode="markers", name="Bottom Up"))
fig.update_layout(title="Time to Solve (Top Down vs Bottom Up)", xaxis_title="Array Size", yaxis_title="Time (us)")

fig.show()

In [32]:
# clean the data with rolling average
df['bottomclean'] = df['Bottom Up Time (us)'].rolling(window=30).mean()
df['topclean'] = df['Top Down Time (us)'].rolling(window=30).mean()

In [51]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=df["Array Size"], y=df["topclean"], mode="markers", name="Top Down"))
fig.add_trace(go.Scatter(x=df["Array Size"], y=df["bottomclean"], mode="markers", name="Bottom Up"))
fig.update_layout(title="30 Rolling average applied", xaxis_title="Array Size", yaxis_title="Key Comparisons")

fig.show()

In [34]:
# calculate performance difference between the two algorithms w.r.t. bottom up

df['pct'] = ((df['topclean'] - df['bottomclean'])/df['bottomclean']).rolling(30).median()
df['pct']

0           NaN
1           NaN
2           NaN
3           NaN
4           NaN
         ...   
995    0.374326
996    0.374326
997    0.374326
998    0.374326
999    0.374326
Name: pct, Length: 1000, dtype: float64

In [52]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df["Array Size"],
    y=df["pct"],
    name="pct diff",
    fill='tozeroy',  # Fills the area under the line
    mode='lines',    # Plots as a continuous line
    line_color='red' # Set the color of the area to red
))

fig.update_layout(title='Percentage Difference', xaxis_title='Array Size', yaxis_title='Percentage Difference')

fig.show()

In [55]:
# draw botplot and histogram
fig = go.Figure()

fig.add_trace(go.Box(x=df['pct'], name='pct'))
# add vertical line at mean
# Add vertical line at the mean
# fig.add_trace(go.Scatter(
#     x=[df['pct'].mean(), df['pct'].mean()],
#     y=[0, 1],  # Extend y-axis for the vertical line
#     mode='lines',
#     line=dict(color='red', dash='dash'),
#     name='Mean'
# ))

fig.update_layout(title='Performance Improvement from Top Down to Bottom Up', xaxis_title='Percentage', yaxis_title='Frequency')

fig.show()

In [45]:
fig = go.Figure()

fig.add_trace(go.Histogram(x=df['pct'], nbinsx=12))